In [8]:
# ==============================================================================
# NOTEBOOK 2: PRÉTRAITEMENT TEXTUEL AVANCÉ (PREPROCESSING)
# PROJET: Text Mining - L'Enquêteur du Niger
# ==============================================================================

import pandas as pd
import re
import os
import nltk
from nltk.corpus import stopwords

# Téléchargement des stop-words de base si nécessaire
nltk.download('stopwords', quiet=True)

# 1. Configuration des chemins
INPUT_PATH = "../data/processed/enqueteur_cleaned.csv"
OUTPUT_PATH = "../data/processed/enqueteur_preprocessed.csv"

df = pd.read_csv(INPUT_PATH)
print(f"Dataset chargé : {df.shape[0]} lignes.")


Dataset chargé : 339 lignes.


In [9]:
# ==============================================================================
# ÉTAPE 1 : EXTRACTION DES HASHTAGS (Indicateurs clés du combat)
# ==============================================================================
def extraire_hashtags(text):
    if pd.isna(text):
        return ""
    hashtags = re.findall(r"#\w+", text)
    return " ".join(hashtags).lower()

df['hashtags'] = df['texte'].apply(extraire_hashtags)


In [10]:

# ==============================================================================
# ÉTAPE 2 : NETTOYAGE CHIRURGICAL DU TEXTE
# =============================================================================
def clean_political_text_v2(text):
    if pd.isna(text):
        return ""
    
    # 1. Suppression spécifique de la signature éditoriale répétitive au début du texte
    # Cette regex cherche "Éditorial de [Noms]...", "Quotidien L'Enquêteur du [Date]..."
    # Elle gère les variations de casse (majuscules/minuscules) et les accents.
    pattern_editorial = r"^\s*éditorial\s+de\s+[^,]+,\s+quotidien\s+l[\s’']enqu[êe]teur\s+du\s+\w+\s+\d+\s+\w+\s+\d{4}\s*"
    text_clean = re.sub(pattern_editorial, "", text, flags=re.IGNORECASE)
    
    # Au cas où la structure varie légèrement, on fait sauter les résidus courants d'en-tête
    text_clean = re.sub(r"editorial de soumana idrissa maïga", "", text_clean, flags=re.IGNORECASE)
    text_clean = re.sub(r"quotidien l['’]enqueteur", "", text_clean, flags=re.IGNORECASE)
    
    # 2. Passage en minuscules
    text_clean = text_clean.lower()
    
    # 3. Suppression des URLs Facebook et classiques
    text_clean = re.sub(r'https?://\s*\S+|www\.\S+', '', text_clean)
    text_clean = re.sub(r'fb\.watch/\S+', '', text_clean)
    
    # 4. Suppression des mentions (@quelquun)
    text_clean = re.sub(r'@\w+', '', text_clean)
    
    # 5. Caractères spéciaux (on garde les ponctuations expressives d'indignation : ! et ?)
    text_clean = re.sub(r'[^\w\s\?!]', ' ', text_clean)
    
    # 6. Normalisation des espaces
    text_clean = re.sub(r'\s+', ' ', text_clean).strip()
    
    return text_clean

# Application du nettoyage de surface amélioré
df['texte_nettoye'] = df['texte'].apply(clean_political_text_v2)

In [11]:
# ==============================================================================
# ÉTAPE 3 : GESTION PERSONNALISÉE DES STOP-WORDS (MOTS VIDES)
# ==============================================================================
stop_words_fr = set(stopwords.words('french'))

# On garde toujours les négations indispensables pour le sentiment
mots_a_garder = {'ne', 'pas', 'plus', 'jamais', 'rien', 'aucun', 'contre', 'sans', 'mais', 'non'}
stop_words_fr = stop_words_fr - mots_a_garder

# Stop-words spécifiques : Journaliste, Journal, Jours de la semaine, Mois de l'année
mots_calendrier_et_signatures = {
    # Noms propres de l'éditeur / journal
    'soumana', 'idrissa', 'maïga', 'maiga', 'enqueteur', 'quotidien', 'editorial', 'éditorial','enquêteur', 'cette',
    # Jours
    'lundi', 'mardi', 'mercredi', 'jeudi', 'vendredi', 'samedi', 'dimanche',
    # Mois
    'janvier', 'février', 'mars', 'avril', 'mai', 'juin', 'juillet', 'août', 'septembre', 'octobre', 'novembre', 'décembre',
    # Bruit Facebook / Presse
    'les', 'des', 'plus', 'comme', 'ici', 'cliquez', 'lien', 'vidéo', 'partager', 'page', 'suivre', 'direct'
}

stop_words_final = stop_words_fr.union(mots_calendrier_et_signatures)

def supprimer_stop_words(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words_final]
    return " ".join(filtered_words)

df['texte_tokenise'] = df['texte_nettoye'].apply(supprimer_stop_words)

In [12]:
# ==============================================================================
# INSPECTION VISUELLE DU RÉSULTAT
# ==============================================================================
print("\n--- Aperçu du traitement sur le premier article ---")
print(f"ORIGINAL :\n{df['texte'].iloc[0][:200]}...\n")
print(f"NETTOYÉ & FILTRÉ :\n{df['texte_tokenise'].iloc[0][:200]}...\n")
print(f"HASHTAGS EXTRAITS : {df['hashtags'].iloc[0]}")



--- Aperçu du traitement sur le premier article ---
ORIGINAL :
Les rentiers de la Refondation : quand le "soutien" devient un fonds de commerce
(Editorial de Soumana Idrissa Maïga, Quotidien L’Enquêteur du Mercredi 15 Juillet 2026)
Il est né sous nos yeux une nou...

NETTOYÉ & FILTRÉ :
rentiers refondation quand soutien devient fonds commerce 15 2026 né sous yeux nouvelle profession sans doute lucrative moins exigeante heure rentier refondation kit démarrage portée tous smartphone c...

HASHTAGS EXTRAITS : 


In [13]:
# Sauvegarde pour la phase suivante (Analyse Exploratoire & Sentiments)
df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
print(f"\nDonnées prétraitées sauvegardées dans : {OUTPUT_PATH}")


Données prétraitées sauvegardées dans : ../data/processed/enqueteur_preprocessed.csv
